## Celula 1: Configuracao

Ajuste as variaveis abaixo e execute esta celula primeiro.

In [ ]:

# @title
# ================================================================
# COMFYQUEUE WORKER — Celula 1: Configuracao
# ================================================================
# Ajuste as variaveis abaixo e execute esta celula primeiro.
# ================================================================

from google.colab import drive
drive.mount('/content/drive')

# 2. Caminhos (Importante: Usar aspas se for usar no shell)
DRIVE_COMFYUI_PATH = "/content/drive/Shared drives/IA/ComfyUI"
COMFYUI_REPO_PATH = "/content/ComfyUI"
COMFYUI_PORT = 8188

LARAVEL_API_URL = "https://apps.spigo.net"  # URL base do servidor Laravel (sem barra final)
DEFAULT_CHECKPOINT = ""  # opcional: ex. "v1-5-pruned-emaonly-fp16.safetensors"

# Tempo maximo de espera por job (em segundos). Aumente se os jobs forem demorados.
COMFYUI_TIMEOUT = 3600  # 1 hora

# Headers para requests HTTP ao Laravel
HEADERS = {
    "Accept": "application/json",
    "Content-Type": "application/json",
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0 Safari/537.36",
}

# Configuracao opcional de upload via SFTP
SFTP_HOST = ""
SFTP_PORT = 22
SFTP_USERNAME = ""
SFTP_PASSWORD = ""

# Configuracao de upload via FTP simples (alternativa ao SFTP)
FTP_HOST = "spigo.net"
FTP_PORT = 21
FTP_USERNAME = "apps.comfy@spigo.net"
FTP_PASSWORD = "J$y@H6Rat@$d"
FTP_USE_TLS = False
FTP_PASSIVE = True
FTP_BASE_DIR = ""

print("Configuracao carregada")
print(f"  Servidor Laravel : {LARAVEL_API_URL}")
print(f"  ComfyUI repo     : {COMFYUI_REPO_PATH}")
print(f"  ComfyUI drive    : {DRIVE_COMFYUI_PATH}")
print(f"  Porta ComfyUI    : {COMFYUI_PORT}")
print(f"  Checkpoint fixo  : {DEFAULT_CHECKPOINT or '[auto]'}")
print(f"  Timeout por job  : {COMFYUI_TIMEOUT}s")
print(f"  FTP host         : {FTP_HOST}:{FTP_PORT}")
print(f"  FTP user         : {FTP_USERNAME}")
print(f"  FTP base dir     : {FTP_BASE_DIR}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Configuracao carregada
  Servidor Laravel : https://apps.spigo.net
  ComfyUI repo     : /content/ComfyUI
  ComfyUI drive    : /content/drive/Shared drives/IA/ComfyUI
  Porta ComfyUI    : 8188
  Checkpoint fixo  : [auto]
  SFTP host        : ftp.spigo.net:21
  SFTP user        : apps.comfy@spigo.net
  FTP host         : spigo.net:21
  FTP user         : apps.comfy@spigo.net
  FTP base dir     : 


## Celula 2 - Instalar ComfyUI

Clona (ou atualiza) o repositorio e instala as dependencias Python. Tambem prepara links simbolicos para usar os modelos armazenados no Google Drive.

In [ ]:
# @title
import os, subprocess, sys
from pathlib import Path

def run(cmd, **kwargs):
    """Executa um comando shell e gera excecao se falhar."""
    subprocess.run(cmd, shell=True, check=True, **kwargs)

# Clonar ou atualizar ComfyUI repo local
if not os.path.exists(COMFYUI_REPO_PATH):
    print("Clonando ComfyUI...")
    run(f"git clone --depth 1 https://github.com/comfyanonymous/ComfyUI {COMFYUI_REPO_PATH}")
else:
    print("Atualizando ComfyUI...")
    run("git pull", cwd=COMFYUI_REPO_PATH)

# Instalar dependencias Python
print("Instalando dependencias...")
run(f"{sys.executable} -m pip install -r {COMFYUI_REPO_PATH}/requirements.txt -q")

# Instalar xformers: melhora velocidade de cross-attention ~15-20% nas GPUs Ampere/Volta
print("Instalando xformers...")
run(f"{sys.executable} -m pip install xformers --upgrade -q")

# Garantir pastas persistentes no Drive
Path(DRIVE_COMFYUI_PATH).mkdir(parents=True, exist_ok=True)
Path(f"{DRIVE_COMFYUI_PATH}/models").mkdir(parents=True, exist_ok=True)
Path(f"{DRIVE_COMFYUI_PATH}/output").mkdir(parents=True, exist_ok=True)
Path(f"{DRIVE_COMFYUI_PATH}/input").mkdir(parents=True, exist_ok=True)

# Apontar repo local para usar modelos/output/input no Drive
for folder in ["models", "output", "input"]:
    local_path = Path(COMFYUI_REPO_PATH) / folder
    drive_path = Path(DRIVE_COMFYUI_PATH) / folder

    if local_path.exists() and not local_path.is_symlink():
        # Preserva conteudo original no primeiro setup
        backup = Path(COMFYUI_REPO_PATH) / f"{folder}_backup"
        if not backup.exists():
            local_path.rename(backup)
        else:
            run(f"rm -rf '{local_path}'")

    if local_path.is_symlink() and os.readlink(local_path) != str(drive_path):
        local_path.unlink()

    if not local_path.exists():
        local_path.symlink_to(drive_path, target_is_directory=True)

print("\nComfyUI instalado e configurado para usar modelos/output/input no Drive")

# Verificar GPU disponivel
gpu_info = subprocess.run(
    "nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader",
    shell=True, capture_output=True, text=True,
)
if gpu_info.returncode == 0:
    print(f"GPU: {gpu_info.stdout.strip()}")
else:
    print("AVISO: GPU nao detectada. Verifique o runtime (Runtime > Change runtime type > GPU).")


Atualizando ComfyUI...
Instalando dependencias...

ComfyUI instalado e configurado para usar modelos no Drive


## Célula 3 — Baixar modelos

Busca os modelos necessários dos jobs **pendentes** no Laravel e faz download apenas dos que ainda não existem no ComfyUI.

In [ ]:
# @title
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path


def _download_single_model(model: dict) -> str:
    """Baixa um modelo e retorna mensagem de status. Executado em thread."""
    name = model.get("name", "")
    dest = model.get("dest", "models/checkpoints")
    url  = model.get("url", "")

    if not name or not url:
        return f"  Entrada invalida (sem nome ou URL): {model}"

    clean_dest = dest.removeprefix("/").removeprefix("models/")
    dest_dir  = Path(DRIVE_COMFYUI_PATH) / "models" / clean_dest
    dest_file = dest_dir / name
    dest_dir.mkdir(parents=True, exist_ok=True)

    if dest_file.exists():
        return f"  {name} - ja existe ({dest_file.stat().st_size / 1024 / 1024:.0f} MB), pulando"

    with requests.get(url, stream=True, timeout=3600) as r:
        r.raise_for_status()
        with open(dest_file, "wb") as f:
            for chunk in r.iter_content(chunk_size=2 * 1024 * 1024):
                if chunk:
                    f.write(chunk)

    return f"  {name} - concluido ({dest_file.stat().st_size / 1024 / 1024:.0f} MB)"


def download_models():
    resp = requests.get(f"{LARAVEL_API_URL}/api/comfy-queue/models", headers=HEADERS, timeout=30)
    resp.raise_for_status()

    models = resp.json()
    if not models:
        print("Nenhum modelo necessario para os jobs pendentes.")
        return

    print(f"{len(models)} modelo(s) necessario(s):\n")

    # Downloads paralelos — max 2 simultaneos para nao saturar a rede/Drive
    with ThreadPoolExecutor(max_workers=2) as executor:
        futures = {executor.submit(_download_single_model, m): m for m in models}
        for future in as_completed(futures):
            try:
                print(future.result())
            except Exception as exc:
                name = futures[future].get("name", "?")
                print(f"  {name} - ERRO: {exc}")

    print("\nModelos prontos no Drive")


download_models()


Nenhum modelo necessario para os jobs pendentes.


## Célula 4 — Executar jobs

Inicia o ComfyUI, busca os jobs pendentes um a um, envia cada workflow ao ComfyUI e reporta o resultado (sucesso ou erro) de volta ao Laravel.

In [ ]:

# @title
import json
import mimetypes
import posixpath
import requests, subprocess, sys, time
from pathlib import Path

COMFYUI_BASE = f"http://127.0.0.1:{COMFYUI_PORT}"
COMFYUI_LOG_PATH = Path("/content/comfyui.log")
GET_HEADERS = {k: v for k, v in HEADERS.items() if k.lower() != "content-type"}


class ComfyUIUnavailableError(RuntimeError):
    """Representa indisponibilidade do processo HTTP do ComfyUI."""


# Utilitarios

def ensure_paramiko():
    """Garante que o pacote paramiko esteja disponivel no Colab."""
    try:
        import paramiko  # noqa: F401
    except Exception:
        subprocess.run([sys.executable, "-m", "pip", "install", "paramiko", "-q"], check=True)


def read_comfyui_log_tail(max_chars: int = 2500) -> str:
    """Le as ultimas linhas do log do ComfyUI para diagnostico."""
    try:
        content = COMFYUI_LOG_PATH.read_text(encoding="utf-8", errors="ignore")
    except FileNotFoundError:
        return "<log do ComfyUI indisponivel>"

    content = content.strip()
    if not content:
        return "<log do ComfyUI vazio>"

    if len(content) > max_chars:
        content = content[-max_chars:]

    return content


def stop_comfyui(proc):
    """Encerra o processo do ComfyUI e fecha o arquivo de log."""
    if proc is None:
        return

    try:
        if proc.poll() is None:
            proc.terminate()
            try:
                proc.wait(timeout=15)
            except subprocess.TimeoutExpired:
                proc.kill()
                proc.wait(timeout=5)
    finally:
        log_handle = getattr(proc, "log_handle", None)
        if log_handle and not log_handle.closed:
            log_handle.close()


def build_comfyui_unavailable_message(action: str, proc=None, exc: Exception | None = None) -> str:
    """Monta uma mensagem rica para quando o ComfyUI fica indisponivel."""
    details = [f"ComfyUI indisponivel durante {action}."]

    if proc is not None and proc.poll() is not None:
        details.append(f"Processo encerrado com codigo {proc.returncode}.")

    if exc is not None:
        details.append(f"Erro HTTP: {exc}")

    log_tail = read_comfyui_log_tail()
    if log_tail:
        details.append("Ultimas linhas do log:")
        details.append(log_tail)

    return "\n".join(details)


def start_comfyui():
    """Inicia o processo do ComfyUI e aguarda ficar disponivel (ate 3 min)."""
    print("Iniciando ComfyUI...")
    COMFYUI_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
    log_handle = open(COMFYUI_LOG_PATH, "w", encoding="utf-8")

    proc = subprocess.Popen(
        [sys.executable, "main.py", "--port", str(COMFYUI_PORT), "--listen", "--disable-auto-launch", "--fast", "--preview-method", "none"],
        cwd=COMFYUI_REPO_PATH,
        stdout=log_handle,
        stderr=subprocess.STDOUT,
        text=True,
    )
    proc.log_handle = log_handle

    for attempt in range(60):
        if proc.poll() is not None:
            stop_comfyui(proc)
            raise ComfyUIUnavailableError(build_comfyui_unavailable_message("a inicializacao", proc=proc))

        time.sleep(3)
        try:
            r = requests.get(f"{COMFYUI_BASE}/system_stats", timeout=3)
            if r.status_code == 200:
                print(f"ComfyUI disponivel (tentativa {attempt + 1})")
                print(f"Log do ComfyUI: {COMFYUI_LOG_PATH}")
                return proc
        except requests.exceptions.RequestException:
            pass

    stop_comfyui(proc)
    raise ComfyUIUnavailableError(build_comfyui_unavailable_message("a inicializacao", proc=proc))


def restart_comfyui(proc, reason: str):
    """Reinicia o ComfyUI apos indisponibilidade detectada."""
    print(f"Reiniciando ComfyUI ({reason})...")
    stop_comfyui(proc)
    return start_comfyui()


def ensure_comfyui_available(proc, action: str):
    """Valida se o processo e a API do ComfyUI seguem respondendo."""
    if proc is not None and proc.poll() is not None:
        raise ComfyUIUnavailableError(build_comfyui_unavailable_message(action, proc=proc))

    try:
        r = requests.get(f"{COMFYUI_BASE}/system_stats", timeout=5)
        r.raise_for_status()
    except requests.exceptions.RequestException as exc:
        raise ComfyUIUnavailableError(build_comfyui_unavailable_message(action, proc=proc, exc=exc)) from exc


def download_job_inputs(job: dict):
    """Baixa os arquivos de input do job para a pasta input do ComfyUI."""
    input_files = job.get("input_files") or []
    if not input_files:
        return

    input_dir = Path(DRIVE_COMFYUI_PATH) / "input"
    input_dir.mkdir(parents=True, exist_ok=True)

    print(f"  {len(input_files)} arquivo(s) de input para baixar:")
    for entry in input_files:
        url = entry.get("url", "")
        if not url:
            print(f"    Entrada sem URL, pulando: {entry}")
            continue

        name = entry.get("name") or url.split("?")[0].rstrip("/").split("/")[-1]
        dest = input_dir / name

        if dest.exists():
            print(f"    {name} - ja existe, pulando")
            continue

        print(f"    Baixando {name}...")
        with requests.get(url, stream=True, timeout=3600) as r:
            r.raise_for_status()
            with open(dest, "wb") as f:
                for chunk in r.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        f.write(chunk)
        print(f"    {name} - concluido ({dest.stat().st_size} bytes)")


def detect_checkpoint_name(job: dict) -> str:
    """Resolve qual checkpoint usar para jobs txt2img simplificados."""
    params = job.get("params") or {}
    required_models = job.get("required_models") or []

    explicit = (
        params.get("checkpoint")
        or params.get("ckpt_name")
        or params.get("model")
        or DEFAULT_CHECKPOINT
    )
    if explicit:
        return explicit

    for model in required_models:
        dest = (model.get("dest") or "").strip("/")
        name = model.get("name")
        if name and (dest == "models/checkpoints" or dest.endswith("/checkpoints") or dest == "checkpoints"):
            return name

    checkpoints_dir = Path(DRIVE_COMFYUI_PATH) / "models" / "checkpoints"
    if checkpoints_dir.exists():
        candidates = sorted(
            [path.name for path in checkpoints_dir.iterdir() if path.is_file() and path.suffix.lower() in {".safetensors", ".ckpt", ".pt", ".pth"}]
        )
        if candidates:
            return candidates[0]

    raise RuntimeError(
        "Nenhum checkpoint encontrado. Defina DEFAULT_CHECKPOINT, informe 'checkpoint' no job, "
        "ou cadastre um required_model em models/checkpoints."
    )


def build_txt2img_workflow(job: dict) -> dict:
    """Converte o formato simplificado txt2img em workflow API do ComfyUI."""
    params = job.get("params") or {}
    checkpoint_name = detect_checkpoint_name(job)
    seed = int(params.get("seed", -1))
    if seed < 0:
        seed = int(time.time() * 1000) % 18446744073709551615

    width = int(params.get("width", 512))
    height = int(params.get("height", 512))
    steps = int(params.get("steps", 20))
    cfg_scale = float(params.get("cfg_scale", 7.0))
    sampler_name = params.get("sampler_name") or params.get("sampler") or "euler"
    scheduler = params.get("scheduler", "normal")
    prompt = params.get("prompt", "")
    negative_prompt = params.get("negative_prompt", "")

    return {
        "4": {
            "class_type": "CheckpointLoaderSimple",
            "inputs": {
                "ckpt_name": checkpoint_name,
            },
        },
        "5": {
            "class_type": "EmptyLatentImage",
            "inputs": {
                "width": width,
                "height": height,
                "batch_size": 1,
            },
        },
        "6": {
            "class_type": "CLIPTextEncode",
            "inputs": {
                "text": prompt,
                "clip": ["4", 1],
            },
        },
        "7": {
            "class_type": "CLIPTextEncode",
            "inputs": {
                "text": negative_prompt,
                "clip": ["4", 1],
            },
        },
        "8": {
            "class_type": "KSampler",
            "inputs": {
                "seed": seed,
                "steps": steps,
                "cfg": cfg_scale,
                "sampler_name": sampler_name,
                "scheduler": scheduler,
                "denoise": 1,
                "model": ["4", 0],
                "positive": ["6", 0],
                "negative": ["7", 0],
                "latent_image": ["5", 0],
            },
        },
        "9": {
            "class_type": "VAEDecode",
            "inputs": {
                "samples": ["8", 0],
                "vae": ["4", 2],
            },
        },
        "10": {
            "class_type": "SaveImage",
            "inputs": {
                "filename_prefix": f"ComfyQueue/job_{job['id']}",
                "images": ["9", 0],
            },
        },
    }


def normalize_workflow(job: dict) -> dict:
    """Aceita workflow API completo ou formato simplificado txt2img."""
    params = job.get("params") or {}

    if isinstance(params, dict) and any(str(key).isdigit() for key in params.keys()):
        return params

    if job.get("type") == "txt2img":
        return build_txt2img_workflow(job)

    raise RuntimeError(
        "Formato de job nao suportado. Envie um workflow API do ComfyUI "
        "ou use type='txt2img' com os campos simplificados."
    )


def submit_workflow(workflow: dict) -> str:
    """Envia o workflow ao ComfyUI e retorna o prompt_id."""
    r = requests.post(f"{COMFYUI_BASE}/prompt", json={"prompt": workflow}, timeout=30)
    if r.status_code >= 400:
        raise RuntimeError(f"Falha ao enviar workflow ao ComfyUI: {r.status_code} - {r.text[:500]}")
    return r.json()["prompt_id"]


def wait_for_result(prompt_id: str, proc, timeout_sec: int = COMFYUI_TIMEOUT) -> dict:
    """Aguarda a conclusao do prompt e retorna o historico."""
    deadline = time.time() + timeout_sec
    while time.time() < deadline:
        if proc is not None and proc.poll() is not None:
            raise ComfyUIUnavailableError(build_comfyui_unavailable_message(f"a execucao do prompt {prompt_id}", proc=proc))

        time.sleep(5)
        try:
            r = requests.get(f"{COMFYUI_BASE}/history/{prompt_id}", timeout=10)
        except requests.exceptions.RequestException as exc:
            raise ComfyUIUnavailableError(build_comfyui_unavailable_message(f"a consulta do historico do prompt {prompt_id}", proc=proc, exc=exc)) from exc

        if r.status_code == 200:
            history = r.json()
            if prompt_id in history:
                return history[prompt_id]

    raise TimeoutError(f"Job {prompt_id} nao concluido em {timeout_sec}s")


def extract_output_files(history: dict) -> list:
    """Extrai a lista de arquivos gerados do historico do ComfyUI."""
    files = []
    for node_output in history.get("outputs", {}).values():
        for img in node_output.get("images", []):
            files.append({
                "filename": img["filename"],
                "subfolder": img.get("subfolder", ""),
                "type": img.get("type", "output"),
            })
    return files


def resolve_output_path(file_info: dict) -> Path | None:
    """Resolve o caminho local do arquivo gerado pelo ComfyUI para upload."""
    filename = file_info.get("filename")
    if not filename:
        return None

    subfolder = (file_info.get("subfolder") or "").strip("/")
    candidates = [
        Path(DRIVE_COMFYUI_PATH) / "output" / subfolder / filename,
        Path(COMFYUI_REPO_PATH) / "output" / subfolder / filename,
    ]

    for path in candidates:
        if path.exists() and path.is_file():
            return path

    return None


def validate_sftp_config():
    """Valida configuracoes obrigatorias para upload SFTP."""
    missing = []
    if not SFTP_HOST:
        missing.append("SFTP_HOST")
    if not SFTP_USERNAME:
        missing.append("SFTP_USERNAME")
    if not SFTP_PASSWORD:
        missing.append("SFTP_PASSWORD")

    if missing:
        raise RuntimeError("Configure as variaveis SFTP na Celula 1: " + ", ".join(missing))


def sftp_ensure_dir(sftp_client, remote_dir: str):
    """Cria diretorio remoto recursivamente se ainda nao existir."""
    parts = [p for p in remote_dir.split("/") if p]
    current = ""
    for part in parts:
        current = f"{current}/{part}" if current else part
        try:
            sftp_client.stat(current)
        except FileNotFoundError:
            sftp_client.mkdir(current)


def upload_file_via_sftp(job_id: int, local_path: Path) -> dict:
    """Faz upload de um arquivo para a raiz SFTP (chroot) em subpasta do job."""
    import paramiko

    remote_dir = str(job_id)
    remote_filename = local_path.name
    remote_path = posixpath.join(remote_dir, remote_filename)

    transport = paramiko.Transport((SFTP_HOST, int(SFTP_PORT)))
    try:
        transport.connect(username=SFTP_USERNAME, password=SFTP_PASSWORD)
        sftp = paramiko.SFTPClient.from_transport(transport)
        try:
            sftp_ensure_dir(sftp, remote_dir)
            sftp.put(local_path.as_posix(), remote_path)
        finally:
            sftp.close()
    finally:
        transport.close()

    mime = mimetypes.guess_type(local_path.name)[0] or "application/octet-stream"
    public_url = f"{LARAVEL_API_URL}/storage/comfy-queue/jobs/{job_id}/{remote_filename}"

    return {
        "original_name": local_path.name,
        "filename": remote_filename,
        "path": f"comfy-queue/jobs/{job_id}/{remote_filename}",
        "url": public_url,
        "mime": mime,
        "size": local_path.stat().st_size,
        "type": "uploaded",
    }


def upload_outputs_via_sftp(job_id: int, output_files: list) -> list:
    """Envia os arquivos de output via SFTP e retorna metadados para o Laravel."""
    uploaded = []
    for file_info in output_files:
        local_path = resolve_output_path(file_info)
        if not local_path:
            continue
        uploaded.append(upload_file_via_sftp(job_id, local_path))
    return uploaded


def validate_ftp_config():
    """Valida configuracoes obrigatorias para upload FTP."""
    missing = []
    if not FTP_HOST:
        missing.append("FTP_HOST")
    if not FTP_USERNAME:
        missing.append("FTP_USERNAME")
    if not FTP_PASSWORD:
        missing.append("FTP_PASSWORD")
    if missing:
        raise RuntimeError("Configure as variaveis FTP na Celula 1: " + ", ".join(missing))


def ftp_ensure_dir(ftp_client, remote_dir: str):
    """Cria diretorio remoto recursivamente se ainda nao existir via FTP."""
    parts = [p for p in remote_dir.split("/") if p]
    current = ""
    for part in parts:
        current = f"{current}/{part}" if current else part
        try:
            ftp_client.cwd(current)
        except Exception:
            try:
                ftp_client.mkd(current)
                ftp_client.cwd(current)
            except Exception:
                pass
    ftp_client.cwd("/")


def upload_file_via_ftp(job_id: int, local_path: Path) -> dict:
    """Faz upload de um arquivo via FTP simples para o servidor."""
    import ftplib

    remote_dir = f"{FTP_BASE_DIR}/{job_id}"
    remote_filename = local_path.name

    ftp = ftplib.FTP()
    ftp.connect(FTP_HOST, int(FTP_PORT), timeout=30)
    ftp.login(FTP_USERNAME, FTP_PASSWORD)
    ftp.set_pasv(FTP_PASSIVE)

    try:
        ftp_ensure_dir(ftp, remote_dir)
        ftp.cwd(remote_dir)
        with open(local_path.as_posix(), "rb") as f:
            ftp.storbinary(f"STOR {remote_filename}", f)
        ftp.quit()
    except Exception as e:
        try:
            ftp.quit()
        except Exception:
            pass
        raise RuntimeError(f"Falha no upload FTP: {e}")

    mime = mimetypes.guess_type(local_path.name)[0] or "application/octet-stream"
    public_url = f"{LARAVEL_API_URL}/storage/comfy-queue/jobs/{job_id}/{remote_filename}"

    return {
        "original_name": local_path.name,
        "filename": remote_filename,
        "path": f"comfy-queue/jobs/{job_id}/{remote_filename}",
        "url": public_url,
        "mime": mime,
        "size": local_path.stat().st_size,
        "type": "uploaded",
    }


def upload_outputs_via_ftp(job_id: int, output_files: list) -> list:
    """Envia os arquivos de output via FTP e retorna metadados para o Laravel."""
    uploaded = []
    for file_info in output_files:
        local_path = resolve_output_path(file_info)
        if not local_path:
            continue
        uploaded.append(upload_file_via_ftp(job_id, local_path))
    return uploaded


def report_done_via_get(job_id: int, prompt_id: str, output_files: list):
    """Marca job como concluido via GET para contornar bloqueio do WAF no POST."""
    r = requests.get(
        f"{LARAVEL_API_URL}/api/comfy-queue/job/{job_id}/done",
        headers=GET_HEADERS,
        params={
            "prompt_id": prompt_id,
            "output_files": json.dumps(output_files, ensure_ascii=True),
        },
        timeout=60,
    )
    if r.status_code >= 400:
        raise RuntimeError(f"Falha no GET /done: {r.status_code} - {r.text[:500]}")


def report_done(job_id: int, prompt_id: str, output_files: list) -> int:
    """Faz upload FTP e marca done via GET."""
    validate_ftp_config()
    uploaded_files = upload_outputs_via_ftp(job_id, output_files)
    final_outputs = output_files + uploaded_files
    report_done_via_get(job_id, prompt_id, final_outputs)
    return len(uploaded_files)


def report_fail(job_id: int, error_message: str):
    """Reporta falha via GET."""
    r = requests.get(
        f"{LARAVEL_API_URL}/api/comfy-queue/job/{job_id}/fail",
        headers=GET_HEADERS,
        params={"error": error_message[:1500]},
        timeout=30,
    )
    r.raise_for_status()


# Pre-checks (FTP)
validate_ftp_config()

# Loop principal
comfyui_proc = start_comfyui()
processed = 0
failed = 0

try:
    while True:
        resp = requests.get(f"{LARAVEL_API_URL}/api/comfy-queue/next", headers=GET_HEADERS, timeout=30)

        if resp.status_code == 204:
            print("\nNenhum job pendente. Worker finalizado.")
            break

        resp.raise_for_status()
        job = resp.json()
        job_id = job["id"]
        print(f"\n[Job #{job_id}] tipo: {job['type']}")

        try:
            ensure_comfyui_available(comfyui_proc, f"a preparacao do job {job_id}")
            download_job_inputs(job)
            workflow = normalize_workflow(job)
            ensure_comfyui_available(comfyui_proc, f"o envio do job {job_id}")
            prompt_id = submit_workflow(workflow)
            print(f"  prompt_id: {prompt_id}")

            result = wait_for_result(prompt_id, comfyui_proc)
            output_files = extract_output_files(result)
            uploaded_count = report_done(job_id, prompt_id, output_files)

            processed += 1
            print(f"  Concluido - {len(output_files)} arquivo(s), upload(s) ftp: {uploaded_count}")

        except ComfyUIUnavailableError as exc:
            report_fail(job_id, str(exc))
            failed += 1
            print(f"  Falhou: {exc}")
            comfyui_proc = restart_comfyui(comfyui_proc, f"apos indisponibilidade no job {job_id}")

        except Exception as exc:
            report_fail(job_id, str(exc))
            failed += 1
            print(f"  Falhou: {exc}")

finally:
    stop_comfyui(comfyui_proc)
    print(f"\nResumo: {processed} concluido(s), {failed} falha(s)")


Iniciando ComfyUI...
ComfyUI disponivel (tentativa 4)

[Job #423] tipo: img2img
  Falhou: Falha ao enviar workflow ao ComfyUI: 400 - {"error": {"type": "prompt_outputs_failed_validation", "message": "Prompt outputs failed validation", "details": "", "extra_info": {}}, "node_errors": {"41": {"errors": [{"type": "custom_validation_failed", "message": "Custom validation failed for node", "details": "image - Invalid image file: pMofEzLkWBXyBIgHFd6RRgfRndxQlCSEsFYlW5q0.jpg", "extra_info": {"input_name": "image"}}], "dependent_outputs": ["9"], "class_type": "LoadImage"}}}

Nenhum job pendente. Worker finalizado.

Resumo: 0 concluido(s), 1 falha(s)


In [ ]:
from pathlib import Path

pasta = Path("/content/drive/Shared drives/IA/ComfyUI/input")

if not pasta.exists():
    print(f"Pasta não existe: {pasta}")
else:
    arquivos = sorted([p for p in pasta.iterdir() if p.is_file()])
    print(f"Total de arquivos: {len(arquivos)}\n")
    for arq in arquivos:
        print(f"- {arq.name} ({arq.stat().st_size} bytes)")

Total de arquivos: 1

- pMofEzLkWBXyBIgHFd6RRgfRndxQlCSEsFYlW5q0.jpg (194135 bytes)
